In [3]:
import os
from pymongo import MongoClient
import re
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# MongoDB connection details
MONGODB_URI = os.getenv('MONGODB_URI_REALM')
DB_NAME = 'gpt'
COLLECTION_NAME = 'list'

# Connect to MongoDB
client = MongoClient(MONGODB_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

# Define the query
query = {"content": {"$exists": True}, "outline": {"$exists": False}}

# Find documents
documents = collection.find(query)

# Process each document
for doc in documents:
    content = doc.get('content', '')
    # Extract subheadings from the markdown content
    subheadings = re.findall(r'##\s+(.+)', content)
    
    # Remove leading numbers and dots (e.g., "1. ", "2. ", etc.)
    cleaned_subheadings = [re.sub(r'^\d+\.\s*', '', subheading) for subheading in subheadings]
    
    # Print the document ID and cleaned subheadings
    print(f"Document ID: {doc['_id']}")
    print("Extracted Subheadings:")
    for subheading in cleaned_subheadings:
        print(f"- {subheading}")
    print()
    
    # Update the document with the new 'outline' field
    collection.update_one(
        {"_id": doc["_id"]},
        {"$set": {"outline": cleaned_subheadings}}
    )

print("All matching documents have been updated.")

All matching documents have been updated.
